# runtime/01 — Run an agent

## What this demonstrates

End-to-end invocation of a registered agent through the runtime endpoint: `POST /api/v1/runtime/agents/{name}/run`. Verity resolves the agent's champion config, composes the prompts, calls the configured LLM, handles tool-call turns, and writes a full decision-log row to the database. The API returns the resulting `ExecutionResult` synchronously.

**Verity capabilities exercised**

- Config resolution on the server (no client-side pinning).
- Real Anthropic LLM call via the configured inference_config.
- Tool authorization enforcement (agents can only call tools   their version is authorized for).
- Structured decision-log write with message_history,   tool_calls, token usage, and duration.

**A note on cost.** This notebook makes a real Anthropic API call. The appetite_agent used below is configured against `claude-sonnet-4` — a few thousand input tokens per run. Monitor your Anthropic dashboard if you re-run aggressively.

**A note on tools.** The UW business-logic tools (`get_underwriting_guidelines`, `get_submission_context`) are registered as Python callables in the UW demo process — not in the Verity standalone process that serves the REST API. So when we invoke `appetite_agent` from here, the agent's tool calls are attempted but the implementations aren't registered, and the agent returns a graceful explanation. That's expected — it demonstrates that Verity cleanly separates governance (the registry, prompts, tool authorizations) from implementation (where the Python code actually lives).


## Prerequisites

- `ds_workbench` application registered (run `00_setup.ipynb`).
- `appetite_agent` seeded with a champion version (default in   the project's seed data — no action needed).


In [ ]:
import os, sys
HERE = os.getcwd()
if os.path.basename(HERE) != 'ds_workbench':
    for candidate in (os.path.dirname(HERE),
                      os.path.dirname(os.path.dirname(HERE)),
                      '/home/jovyan/work'):
        if os.path.isdir(os.path.join(candidate, 'utility')):
            sys.path.insert(0, candidate); break

from utility.verity import VerityAPI, VerityAPIError
from utility.html import inject_style, badge, render_list, render_detail, render_cards

inject_style()
v = VerityAPI(application='ds_workbench')

In [ ]:
# Confirm the agent exists with a champion we can exercise.
config = v.get_agent_config('appetite_agent')
render_detail(
    'appetite_agent',
    subtitle=f"v{config['version_label']}",
    header_badges=[
        (config.get('lifecycle_state','?'), config.get('lifecycle_state','')),
        (config.get('materiality_tier','?'), config.get('materiality_tier','')),
    ],
    sections=[
        {'title':'Inference config','fields':[
            ('Model',       config['inference_config'].get('model_name')),
            ('Temperature', config['inference_config'].get('temperature')),
            ('Max tokens',  config['inference_config'].get('max_tokens')),
        ]},
        {'title':f"Tools authorized ({len(config.get('tools') or [])})",
         'table':{
             'columns':[('tool_name','Tool'), ('transport','Transport','neutral')],
             'rows': config.get('tools') or [],
         }},
    ],
)

## Execute

Run the agent with a realistic submission context. The call returns synchronously when the agent has finished its agentic loop (all tool turns complete) — which for this run typically takes 5–15 seconds of wall time depending on how many tool attempts the agent makes.


In [ ]:
# A realistic UW submission context — the agent's prompt
# templates substitute these values into {{producer_name}},
# {{lob}}, {{premium_estimate}}, etc.
context = {
    'submission_id':   'notebook-smoke-001',
    'producer_name':   'Acme Specialty Brokers',
    'insured_name':    'Skyline Tech Co.',
    'lob':             'general_liability',
    'industry':        'software',
    'effective_date':  '2026-06-01',
    'premium_estimate': 45000,
}
result = v.run_agent('appetite_agent', context=context)
print(f"decision_log_id : {result['decision_log_id']}")
print(f"status          : {result['status']}")
print(f"tool_calls      : {len(result.get('tool_calls') or [])}")
print(f"input tokens    : {result['input_tokens']}")
print(f"output tokens   : {result['output_tokens']}")
print(f"duration_ms     : {result['duration_ms']}")

## Review results

Two views of the same run:

1. The `ExecutionResult` summary returned by the API call.
2. The full decision-log row Verity persisted — fetched via    `GET /api/v1/decisions/{id}` using the id we just got.


In [ ]:
# Short summary from the execution result.
render_detail(
    'Execution result',
    subtitle=result['decision_log_id'],
    header_badges=[
        (result['status'], result['status']),
        (result['entity_type'], result['entity_type']),
    ],
    sections=[
        {'title':'Identity','fields':[
            ('Entity',  f"{result['entity_type']} / {result['entity_name']}"),
            ('Version', result['version_label']),
            ('Status',  result['status']),
        ]},
        {'title':'Output','fields':[
            ('Summary', result.get('output_summary') or '(empty)'),
        ]},
        {'title':'Timing & usage','fields':[
            ('Duration (ms)',  result['duration_ms']),
            ('Input tokens',   result['input_tokens']),
            ('Output tokens',  result['output_tokens']),
        ]},
        {'title':f"Tool calls attempted ({len(result.get('tool_calls') or [])})",
         'table':{
             'columns':[
                 ('tool_name','Tool'),
                 ('transport','Transport','neutral'),
                 ('status','Status','*'),
             ],
             'rows': result.get('tool_calls') or [],
         }},
    ],
)

In [ ]:
# Full persisted decision row — what compliance reviewers see
# in the admin UI. Carries message_history for audit replay.
decision = v.get_decision(result['decision_log_id'])
render_detail(
    f"Decision — {decision.get('entity_name','?')}",
    subtitle=str(decision.get('id','')),
    header_badges=[
        (decision.get('status','?'), decision.get('status','')),
        (decision.get('entity_type','?'), decision.get('entity_type','')),
    ],
    sections=[
        {'title':'Identity','fields':[
            ('Entity',      f"{decision.get('entity_type')} / {decision.get('entity_name')}"),
            ('Version',     decision.get('version_label')),
            ('Channel',     decision.get('channel')),
            ('Application', decision.get('application')),
            ('Created',     decision.get('created_at')),
        ]},
        {'title':'Timing & usage','fields':[
            ('Duration (ms)', decision.get('duration_ms')),
            ('Input tokens',  decision.get('input_tokens')),
            ('Output tokens', decision.get('output_tokens')),
            ('Model',         decision.get('model_used')),
        ]},
        {'title':f"Tool calls ({len(decision.get('tool_calls_made') or [])})",
         'table':{
             'columns':[
                 ('tool_name','Tool'),
                 ('transport','Transport','neutral'),
                 ('mcp_server_name','MCP server'),
                 ('status','Status','*'),
             ],
             'rows': decision.get('tool_calls_made') or [],
         }},
    ],
)

---

The decision row is now permanent in the audit trail. Its id is visible at `/admin/decisions/{id}` in the admin UI, and can be queried via `GET /api/v1/decisions/{id}` from any external caller. Move on to **`compliance/01_decision_log_walkthrough.ipynb`** to explore the decision-log surface more broadly.
